# Google TimesFM 3.0 与 BigQuery 原生时序大模型零售需求预测

本 Notebook 作为 `retail-demand-forecast` 系列教程的第 36 篇，系统展示如何将 **Google Research TimesFM 3.0**（3.3 亿参数时序基础大模型）与 **Google Cloud BigQuery** 原生托管推理结合，在四大典型零售场景下进行零样本（Zero-Shot）预测、分位数不确定性估计与基准对比。

### 本教程核心涵盖：
1. **BigQuery 原生托管架构**：解析为何无需 `CREATE MODEL`，直接使用 `AI.FORECAST(..., model => 'TimesFM 3.0')` 获得毫秒级预测；
2. **全历史多模型基准对比**：横向对比 TimesFM 3.0 与 Seasonal Naive、调优 Prophet、特征工程 LightGBM；
3. **缺货删失去偏诊断**：量化真实零售缺货截断对基础模型造成的系统性负向偏差；
4. **极短样本冷启动曲线**：检验在 14 天到 1068 天历史上下文下，基础模型的小样本先验优势。


In [ ]:
from pathlib import Path
import sys

# 确保能够加载核心 retail_forecast 包
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from retail_forecast.common import (
    DATA_DIR,
    HORIZON,
    SCENARIOS,
    datasets,
    sales_prediction,
    valid_labels,
)
from retail_forecast.timesfm_benchmark import (
    compute_metrics,
    fit_predict_lightgbm,
    fit_predict_prophet,
    predict_seasonal_naive,
)

RUNS_DIR = ROOT / "runs" / "timesfm-validation-20261103"
print(f"Project Root: {ROOT}")
print(f"Runs Directory: {RUNS_DIR}")

## 1. 零售场景数据加载与探索

我们复用 `notebooks/00-generate-teaching-data.ipynb` 生成的四套墨尔本合成零售场景：
- **Grocery**：高频连续需求，包含 24 个缺货截断日；
- **Wine**：泊松计数脉冲，强促销与年末峰值；
- **Garden**：强年度季节性正弦周期与长周末 DIY；
- **Luxury**：极端稀疏间歇需求（93.6% 零值率）。


In [ ]:
summary_rows = []
for sc in SCENARIOS:
    df = datasets[sc]
    summary_rows.append(
        {
            "Scenario": sc,
            "Total Days": len(df),
            "Train Days": len(df) - HORIZON,
            "Test Days": HORIZON,
            "Mean Sales": df["y"].mean(),
            "Zero Rate (%)": (df["y"] == 0).mean() * 100,
        }
    )
pd.DataFrame(summary_rows)

## 2. BigQuery 原生 TimesFM 3.0 预测结果加载

在云端，我们通过如下标准 BigQuery SQL 语法调用托管的 TimesFM 3.0（完整脚本见 `sql/bigquery_timesfm_experiments.sql`）：

```sql
SELECT
  'grocery' AS scenario,
  'exp1_full_history' AS experiment_id,
  'TimesFM (BigQuery)' AS model,
  DATE(forecast_timestamp) AS ds,
  forecast_value AS prediction,
  prediction_interval_lower_bound AS q10,
  prediction_interval_upper_bound AS q90
FROM AI.FORECAST(
  (
    SELECT ds, y
    FROM `gcp-cc-ai-ml-llm.retail_forecast.grocery_scenario_data`
    WHERE ds <= '2025-12-03' AND is_open = TRUE AND stockout = FALSE
  ),
  model => 'TimesFM 3.0',
  data_col => 'y',
  timestamp_col => 'ds',
  horizon => 28,
  confidence_level => 0.80
);
```

接下来读取从 BigQuery 导出的标准结果集：


In [ ]:
pred_csv = RUNS_DIR / "bq_timesfm_predictions.csv"
df_bq = pd.read_csv(pred_csv, parse_dates=["ds"])
print(f"Total BQ Predictions loaded: {len(df_bq)} rows")
df_bq.head(5)

## 3. 全历史基准实验（Experiment 1: Full History Benchmark）

我们对比四类代表性模型在最后 28 天测试期（2025-12-04 至 2025-12-31）的预测指标：
1. **Seasonal Naive**：7 天周期朴素基线
2. **Prophet**：调优外部假日与促销回归项
3. **LightGBM**：滑窗统计与滞后特征工程
4. **TimesFM (BigQuery)**：零样本（Zero-Shot）时序基础大模型


In [ ]:
benchmark_csv = RUNS_DIR / "exp1_full_history_benchmark.csv"
if benchmark_csv.is_file():
    df_exp1 = pd.read_csv(benchmark_csv)
else:
    # 若本地未缓存，可直接运行脚本生成：uv run python scripts/run_timesfm_benchmark.py
    df_exp1 = pd.DataFrame()
df_exp1

## 4. 预测曲线与 80% 置信区间可视化 (P10–P90)

绘制四大零售场景下的多模型预测对比图：


In [ ]:
fig_path = ROOT / "images" / "tsfm" / "fig-tsfm-01-four-scenarios-comparison.png"
if fig_path.is_file():
    from IPython.display import Image, display

    display(Image(filename=str(fig_path)))
else:
    print(
        "Figure not found. Run `uv run python scripts/run_timesfm_benchmark.py` to generate it."
    )

## 5. 缺货删失去偏诊断（Experiment 2: Stockout Censoring）

在真实零售中，当发生缺货时，观察到的销量被截断在当前库存上限，而非消费者的潜在真实需求。
如果直接将受污染的原始销量暴露给 TimesFM，模型的预测将发生显著的系统性低估：


In [ ]:
exp2_csv = RUNS_DIR / "exp2_stockout_censoring.csv"
if exp2_csv.is_file():
    df_exp2 = pd.read_csv(exp2_csv)
    print(df_exp2.to_string(index=False))

fig_stockout = ROOT / "images" / "tsfm" / "fig-tsfm-03-stockout-censoring-impact.png"
if fig_stockout.is_file():
    from IPython.display import Image, display

    display(Image(filename=str(fig_stockout)))

## 6. 极短样本冷启动退化曲线（Experiment 3: Cold Start Truncation）

对于新开门店或新品上市，企业往往只有 14 ~ 28 天的历史数据。在极短样本下，传统模型的特征滑动窗口会发生空值或周期失真：


In [ ]:
exp3_csv = RUNS_DIR / "exp3_cold_start_curve.csv"
if exp3_csv.is_file():
    df_exp3 = pd.read_csv(exp3_csv)
    # 展示 14 天极限冷启动下的三模型表现
    print(
        df_exp3[df_exp3["Context Length (d)"] == 14][
            ["Scenario", "Model", "WAPE (%)", "MAE"]
        ].to_string(index=False)
    )

fig_cold = ROOT / "images" / "tsfm" / "fig-tsfm-02-coldstart-degradation-curve.png"
if fig_cold.is_file():
    from IPython.display import Image, display

    display(Image(filename=str(fig_cold)))

## 7. 总结与企业级零售落地决策

本实验得出的三项核心工业指导结论：
1. **冷启动与长尾品优先引入基础模型**：在历史长度 $\le 28$ 天时，TimesFM 3.0 的先验常识完全碾压传统机器学习与统计模型；
2. **基础模型不能免除数据去偏**：时序大模型无法自发修正删失截断，必须在特征入湖前完成缺货标记与平滑插补；
3. **In-Database AI 全托管范式**：通过 BigQuery `AI.FORECAST` 直接在数仓内执行批量时序大模型预测，大幅降低了环境依赖与运维成本。

> 详细量化指标与架构设计请参阅报告：[`reports/2026-11-03-timesfm-validation-report.md`](../reports/2026-11-03-timesfm-validation-report.md)。
